# Mafinho Explora — Voice Generator (Colab GPU)

Gera ~532 frases com voice cloning da mae usando XTTS-v2 + GPU T4.

## Instrucoes:
1. **Runtime > Change runtime type > GPU (T4)**
2. Execute cada celula na ordem (Shift+Enter)
3. Faca upload dos arquivos quando solicitado
4. No final, baixe o ZIP com todos os audios

## 1. Instalar dependencias

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

!pip install TTS==0.22.0 pydub "transformers>=4.33.0,<4.46.0" -q
!apt-get install -y ffmpeg -qq
print('Dependencias instaladas!')

## 2. Verificar GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('AVISO: GPU nao detectada!')
    print('Va em Runtime > Change runtime type > GPU')

## 3. Upload dos arquivos

Faca upload de:
- `generate_voices.py` (script de geracao)
- `audiomaeportugues.mp4` (amostra de voz PT)
- `audiomaeingles.mp4` (amostra de voz EN)

In [ ]:
from google.colab import files
import os

# Cria diretorio para amostras
os.makedirs('audiosmae', exist_ok=True)

print('Selecione os 3 arquivos: generate_voices.py + 2 amostras MP4')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.mp4') or fname.endswith('.m4a') or fname.endswith('.wav'):
        dest = os.path.join('audiosmae', fname)
        os.rename(fname, dest)
        print(f'  Movido: {fname} -> {dest}')
    else:
        print(f'  Mantido: {fname}')

# Lista arquivos
print('\nArquivos prontos:')
!ls -la generate_voices.py audiosmae/

## 4. Teste rapido (5 frases)

Valida que o voice cloning funciona antes de rodar tudo.

In [ ]:
!COQUI_TOS_AGREED=1 python generate_voices.py --test

## 5. Ouvir amostras de teste

Confira a qualidade antes de gerar tudo.

In [ ]:
import IPython.display as ipd
import glob

mp3s = sorted(glob.glob('audio/**/*.mp3', recursive=True))[:6]
if not mp3s:
    print('Nenhum MP3 gerado. Execute a celula de teste primeiro.')
else:
    for f in mp3s:
        print(f'\n{f}:')
        ipd.display(ipd.Audio(f))

## 6. Gerar TODAS as frases

~532 frases. Com GPU T4 leva ~20 minutos.

In [ ]:
!COQUI_TOS_AGREED=1 python generate_voices.py

## 7. Empacotar e baixar

In [ ]:
import os

# Conta arquivos gerados
pt_count = len([f for f in os.listdir('audio/pt') if f.endswith('.mp3')]) if os.path.isdir('audio/pt') else 0
en_count = len([f for f in os.listdir('audio/en') if f.endswith('.mp3')]) if os.path.isdir('audio/en') else 0
print(f'Arquivos gerados: {pt_count} PT + {en_count} EN = {pt_count + en_count} total')

# Cria ZIP
!zip -r mafinho_voices.zip audio/
zip_size = os.path.getsize('mafinho_voices.zip') / 1e6
print(f'\nZIP criado: mafinho_voices.zip ({zip_size:.1f} MB)')

# Download
from google.colab import files
files.download('mafinho_voices.zip')

## Proximo passo

1. Extraia `mafinho_voices.zip` na pasta `app_marcelo/`
2. Verifique que `audio/pt/` e `audio/en/` contem os MP3s
3. Verifique que `audio/manifest.json` existe
4. Commit!